In [ ]:
from pathlib import Path
import pickle
import re
import pandas as pd
import os


os.chdir("/home/tochka-sborki-2026-team-029/project")

PROJECT_DIR = Path.cwd()   
PATH_SKUS = PROJECT_DIR / "data" / "skus.pkl"
OUTPUT_DIR = PROJECT_DIR / "data"
PATH_CATEGORIES = OUTPUT_DIR / "cat_df.parquet"
PATH_META = OUTPUT_DIR / "skus_meta.parquet"
PATH_ATTRIBUTES = OUTPUT_DIR / "attrs.parquet"
PATH_BAD_ROWS = OUTPUT_DIR / "bad_rows.parquet"

KV_PATTERN = re.compile(r"^\s*(?P<name>[^:—\-]{2,60}?)\s*[:—\-]\s*(?P<value>.+?)\s*$")

def as_list(value):
    """Приводит одиночный элемент или список к списку."""
    if value is None:
        return []
    if isinstance(value, list):
        return value
    return [value]


def parse_categories(shop):
    """Строит справочник категорий."""
    raw_categories = shop.get("categories", {}).get("category", [])
    raw_categories = as_list(raw_categories)
    rows = []
    for category in raw_categories:
        rows.append({
            "category_id": int(category["@id"]),
            "parent_id": (
                int(category["@parentId"])
                if category.get("@parentId") is not None
                else pd.NA
            ),
            "category_name": category.get("#text"),
        })

    result = pd.DataFrame(rows, columns=["category_id", "parent_id", "category_name"],)
    result["category_id"] = result["category_id"].astype("int64")
    result["parent_id"] = result["parent_id"].astype("Int64")
    return result


def parse_param(param, sku_id, category_id, bad_rows):
    """Преобразует один param в нормализованную запись атрибута."""
    if isinstance(param, dict):
        name = param.get("@name")
        if name is None:
            bad_rows.append({
                "sku_id": sku_id,
                "reason": "dict без @name",
                "raw": repr(param)[:200],
            })
            return []

        value = param.get("#text")
        return [{
            "sku_id": sku_id,
            "categoryId": category_id,
            "attr": str(name),
            "value": None if value is None else str(value),
            "unit": param.get("@unit"),
            "has_unit": param.get("@unit") is not None,
            "is_search": str(name).endswith("_search"),
        }]

    if isinstance(param, str):
        match = KV_PATTERN.match(param)
        if match:
            name = match.group("name").strip()
            return [{
                "sku_id": sku_id,
                "categoryId": category_id,
                "attr": name,
                "value": match.group("value").strip(),
                "unit": None,
                "has_unit": False,
                "is_search": name.endswith("_search"),
            }]

        bad_rows.append({
            "sku_id": sku_id,
            "reason": "строка param без распознанного разделителя",
            "raw": param[:200],
        })
        return []

    bad_rows.append({
        "sku_id": sku_id,
        "reason": f"неожиданный тип param: {type(param).__name__}",
        "raw": repr(param)[:200],
    })
    return []


def parse_offers(shop):
    """Строит метаданные SKU и длинную таблицу атрибутов."""
    offers = as_list(shop.get("offers", {}).get("offer", []))

    meta_rows = []
    attr_rows = []
    bad_rows = []

    for offer in offers:
        try:
            sku_id = int(offer["@id"])
        except (KeyError, TypeError, ValueError):
            bad_rows.append({
                "sku_id": None,
                "reason": "отсутствует или некорректен offer @id",
                "raw": repr(offer)[:200],
            })
            continue

        categories = offer.get("categories")
        category_id = pd.NA

        if isinstance(categories, dict):
            raw_category_id = categories.get("categoryId")
            if raw_category_id is not None:
                try:
                    category_id = int(raw_category_id)
                except (TypeError, ValueError):
                    bad_rows.append({
                        "sku_id": sku_id,
                        "reason": "некорректный categoryId",
                        "raw": repr(raw_category_id)[:200],
                    })

        params = offer.get("param")
        if params is None:
            params = []
        elif isinstance(params, (dict, str)):
            params = [params]
        elif not isinstance(params, list):
            bad_rows.append({
                "sku_id": sku_id,
                "reason": f"неожиданный тип param: {type(params).__name__}",
                "raw": repr(params)[:200],
            })
            params = []

        meta_rows.append({
            "sku_id": sku_id,
            "available": str(offer.get("@available", "")).lower() == "true",
            "vendor": offer.get("vendor"),
            "model": offer.get("model"),
            "name": offer.get("name"),
            "categoryId": category_id,
            "n_params": len(params),
        })

        for param in params:
            attr_rows.extend(
                parse_param(param, sku_id, category_id, bad_rows)
            )

    meta = pd.DataFrame(meta_rows)
    attrs = pd.DataFrame(
        attr_rows,
        columns=[
            "sku_id", "categoryId", "attr", "value",
            "unit", "has_unit", "is_search",
        ],
    )
    bad = pd.DataFrame(
        bad_rows,
        columns=["sku_id", "reason", "raw"],
    )

    if not meta.empty:
        meta["sku_id"] = meta["sku_id"].astype("int64")
        meta["categoryId"] = meta["categoryId"].astype("Int64")
        meta["n_params"] = meta["n_params"].astype("int32")

    if not attrs.empty:
        attrs["sku_id"] = attrs["sku_id"].astype("int64")
        attrs["categoryId"] = attrs["categoryId"].astype("Int64")
        attrs["has_unit"] = attrs["has_unit"].astype(bool)
        attrs["is_search"] = attrs["is_search"].astype(bool)
        attrs["attr"] = attrs["attr"].astype("category")

    return meta, attrs, bad


def validate_data(categories, meta, attrs):
    """Проверяет базовые ограничения до сохранения."""
    if categories["category_id"].duplicated().any():
        raise ValueError("В cat_df обнаружены повторяющиеся category_id")

    if meta["sku_id"].duplicated().any():
        raise ValueError("В skus_meta обнаружены повторяющиеся sku_id")

    valid_category_ids = set(categories["category_id"])
    referenced_ids = set(meta["categoryId"].dropna().astype(int))
    unknown_ids = referenced_ids - valid_category_ids

    print(f"Категорий: {len(categories):,}")
    print(f"SKU: {len(meta):,}")
    print(f"Строк атрибутов: {len(attrs):,}")
    print(f"SKU с неизвестным categoryId: "
          f"{meta['categoryId'].isin(unknown_ids).sum():,}")


def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print(f"Читаю YML-объект: {PATH_SKUS}")
    with PATH_SKUS.open("rb") as file:
        raw = pickle.load(file)

    shop = raw["yml_catalog"]["shop"]
    print("Дата выгрузки:", raw["yml_catalog"].get("@date"))

    categories = parse_categories(shop)
    meta, attrs, bad = parse_offers(shop)
    validate_data(categories, meta, attrs)
    categories.to_parquet(PATH_CATEGORIES, index=False)
    meta.to_parquet(PATH_META, index=False)
    attrs.to_parquet(PATH_ATTRIBUTES, index=False)
    bad.to_parquet(PATH_BAD_ROWS, index=False)

    print("\nСохранено:")
    print(PATH_CATEGORIES)
    print(PATH_META)
    print(PATH_ATTRIBUTES)
    print(PATH_BAD_ROWS)


if __name__ == "__main__":
    main()

Читаю YML-объект: /home/tochka-sborki-2026-team-029/project/data/skus.pkl
